# NumCompute — End-to-End Quickstart Demo

This notebook walks through the full NumCompute toolkit:
1. CSV I/O with missing values
2. Preprocessing (Imputer, StandardScaler, MinMaxScaler, OneHotEncoder)
3. Sort/Search (top-k, quickselect, binary search)
4. Ranking with tie handling and percentiles
5. Descriptive statistics (batch + Welford streaming)
6. Evaluation metrics (classification + regression + ROC/AUC)
7. Finite-difference gradients and Jacobians
8. Pipeline chaining
9. Loop-vs-vectorised benchmarks


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import numcompute as nc
from numcompute import io, preprocessing, sort_search, rank, stats, metrics, optim, pipeline, utils, benchmarking

print('NumCompute', nc.__version__, '— all modules loaded.')

## 1. CSV I/O

In [ ]:
# Load CSV; missing values → NaN
data = io.load_csv('sample_data.csv', skip_header=True)
print('Shape:', data.shape)
print('NaN count per column:', np.sum(np.isnan(data), axis=0))
print('First 5 rows:\n', data[:5])

## 2. Preprocessing

In [ ]:
# Split features / labels
X_raw = data[:, :3]   # age, income, education
y = data[:, 3].astype(int)

# Impute missing values with column mean
imputer = preprocessing.SimpleImputer(strategy='mean')
X_imp = imputer.fit_transform(X_raw)
print('After imputation — NaN count:', np.sum(np.isnan(X_imp)))

# StandardScaler
scaler = preprocessing.StandardScaler()
X_std = scaler.fit_transform(X_imp)
print('Standardised — mean ≈ 0:', np.round(X_std.mean(axis=0), 6))
print('Standardised — std  ≈ 1:', np.round(X_std.std(axis=0), 6))

# OneHotEncoder on integer education column
edu = X_imp[:, 2:3]
ohe = preprocessing.OneHotEncoder()
edu_ohe = ohe.fit_transform(edu)
print('\nOHE shape for education column:', edu_ohe.shape)
print('Feature names:', ohe.get_feature_names_out())

## 3. Sort / Search

In [ ]:
incomes = X_imp[:, 1]

# Top-3 highest incomes
top_vals, top_idx = sort_search.topk(incomes, k=3, largest=True)
print('Top-3 incomes:', top_vals)
print('At rows:', top_idx)

# Quickselect: 2nd highest income
qs = sort_search.quickselect(incomes, k=2, largest=True)
print('\n2nd highest income (quickselect):', qs)

# Binary search
sorted_incomes = np.sort(incomes)
idx, found = sort_search.binary_search(sorted_incomes, 80000.0)
print(f'\nBinary search for 80000: found={found}, insertion index={idx}')

# Multi-key sort: sort by education then income
sorted_data = sort_search.multi_key_sort(X_imp, keys=[2, 1])
print('\nMulti-key sorted (edu ↑, income ↑):\n', sorted_data)

## 4. Ranking & Percentiles

In [ ]:
# Rank ages with different tie methods
ages = X_imp[:, 0]
for method in ('average', 'dense', 'ordinal', 'min', 'max'):
    r = rank.rank(ages, method=method)
    print(f'Rank ({method:8s}): {np.round(r, 1)}')

# Percentiles
print('\nIncome percentiles:')
for q in [25, 50, 75, 90]:
    p = rank.percentile(incomes, q, interpolation='linear')
    print(f'  P{q:2d}: {p:.1f}')

## 5. Descriptive Statistics

In [ ]:
# Batch stats for income
d = stats.describe(incomes)
print('Income summary:')
for k, v in d.items():
    print(f'  {k:<12}: {v}')

# Histogram
counts, edges = stats.histogram(incomes, bins=5)
print('\nIncome histogram:')
for i, c in enumerate(counts):
    bar = '#' * int(c)
    print(f'  {edges[i]:>8.0f}–{edges[i+1]:>8.0f}: {bar} ({c})')

# Welford streaming stats
w = stats.WelfordStats(n_features=3)
for row in X_imp:
    w.update(row)
print('\nWelford mean:', np.round(w.mean_, 2))
print('Welford std: ', np.round(w.std, 2))
print('Batch std:   ', np.round(np.std(X_imp, axis=0), 2))

## 6. Evaluation Metrics

In [ ]:
# Simulate predictions for demo
np.random.seed(42)
y_pred = (np.random.random(len(y)) > 0.4).astype(int)
y_score = np.random.random(len(y))

print('Classification metrics:')
print(f'  Accuracy : {metrics.accuracy(y, y_pred):.3f}')
print(f'  Precision: {metrics.precision(y, y_pred, average="binary"):.3f}')
print(f'  Recall   : {metrics.recall(y, y_pred, average="binary"):.3f}')
print(f'  F1       : {metrics.f1(y, y_pred, average="binary"):.3f}')

cm, labels = metrics.confusion_matrix(y, y_pred)
print('\nConfusion matrix (rows=true, cols=pred):')
print(cm)

# ROC / AUC
fpr, tpr, thresholds = metrics.roc_curve(y, y_score)
a = metrics.auc(fpr, tpr)
print(f'\nROC AUC: {a:.3f}')

# Regression
y_cont = ages
y_pred_cont = y_cont + np.random.normal(0, 5, len(y_cont))
print(f'\nRegression MSE: {metrics.mse(y_cont, y_pred_cont):.2f}')
print(f'Regression R² : {metrics.r2_score(y_cont, y_pred_cont):.3f}')

## 7. Finite-Difference Gradients & Jacobians

In [ ]:
# Scalar function: f(x) = x₀² + 2x₁² + x₀x₁
# Analytical gradient: [2x₀ + x₁,  4x₁ + x₀]
f = lambda x: x[0]**2 + 2*x[1]**2 + x[0]*x[1]
x0 = np.array([1.0, 2.0])
analytical = np.array([2*x0[0] + x0[1], 4*x0[1] + x0[0]])

for method in ('central', 'forward', 'backward'):
    g = optim.grad(f, x0, method=method)
    err = np.max(np.abs(g - analytical))
    print(f'  grad ({method:8s}): {g}  |  max error: {err:.2e}')

# Vector function Jacobian: F(x) = [x₀², x₁², x₀+x₁]
# J = [[2x₀, 0], [0, 2x₁], [1, 1]]
F = lambda x: np.array([x[0]**2, x[1]**2, x[0] + x[1]])
x1 = np.array([2.0, 3.0])
J_analytical = np.array([[2*x1[0], 0], [0, 2*x1[1]], [1, 1]])
J_num = optim.jacobian(F, x1)
print('\nNumerical Jacobian:')
print(np.round(J_num, 6))
print('Analytical Jacobian:')
print(J_analytical)
print('Max error:', np.max(np.abs(J_num - J_analytical)))

## 8. Pipeline Chaining

In [ ]:
# Chain: Imputer → StandardScaler → MinMaxScaler
pipe = pipeline.Pipeline([
    ('impute', preprocessing.SimpleImputer(strategy='mean')),
    ('scale',  preprocessing.StandardScaler()),
    ('norm',   preprocessing.MinMaxScaler()),
])
X_piped = pipe.fit_transform(X_raw)
print('Pipeline output shape:', X_piped.shape)
print('Min per column:', np.round(X_piped.min(axis=0), 6))
print('Max per column:', np.round(X_piped.max(axis=0), 6))

# FeatureUnion: standard-scaled + min-max-scaled side by side
fu = pipeline.FeatureUnion([
    ('std', preprocessing.StandardScaler()),
    ('mm',  preprocessing.MinMaxScaler()),
])
X_fu = fu.fit_transform(X_imp)
print('\nFeatureUnion output shape:', X_fu.shape, '(3 std cols + 3 mm cols)')

## 9. Benchmark: Loop vs. Vectorised

In [ ]:
import time
import numpy as np

N = 500000
rng = np.random.default_rng(0)
big_arr = rng.standard_normal(N)

def loop_sum(a):
    total = 0.0
    for x in a:
        total += x
    return total

def loop_mean(a):
    return loop_sum(a) / len(a)

def loop_std(a):
    m = loop_mean(a)
    var = 0.0
    for x in a:
        var += (x - m) ** 2
    return (var / len(a)) ** 0.5

benchmarks = [
    ('Sum',  loop_sum,  lambda a: np.sum(a)),
    ('Mean', loop_mean, lambda a: np.mean(a)),
    ('Std',  loop_std,  lambda a: np.std(a)),
]

print(f"{'Operation':<8} {'Loop (ms)':>12} {'NumPy (ms)':>12} {'Speedup':>10}")
print('-' * 48)
for name, loop_fn, np_fn in benchmarks:
    r_loop = benchmarking.timeit(loop_fn, big_arr, n_runs=5, warmup=1)
    r_np   = benchmarking.timeit(np_fn,   big_arr, n_runs=10, warmup=2)
    sp     = r_loop['mean_s'] / r_np['mean_s']
    print(f"{name:<8} {r_loop['mean_s']*1000:>11.1f}  {r_np['mean_s']*1000:>11.2f}  {sp:>8.1f}x")

print('\nNote: timings vary by hardware. Results above from this session.')